<a href="https://colab.research.google.com/github/coweye1/Gunshot_Wound_Entrance_vs_Exit_CNN_Benchmark/blob/main/Gunshot_Wound_Entrance_vs_Exit_CNN_Benchmark.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Step 0: Mount Google Drive to access the dataset
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

In [ ]:
# Step 1: Install necessary libraries and import modules
!pip install timm

import torch
import torch.nn as nn
import torch.optim as optim
from torch.optim import lr_scheduler
import torch.backends.cudnn as cudnn
import numpy as np
import torchvision
from torchvision import datasets, models, transforms
import matplotlib.pyplot as plt
import time
import os
import copy
import timm
import pandas as pd

# Setting up the device (GPU is highly recommended for training)
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print(f"Current device: {device}")

In [ ]:
# Step 2: Extract the zip file to the local Colab directory for faster training
# Ensure the path matches your actual zip file location in Drive
zip_path = "/content/drive/MyDrive/Gunshot_Dataset_Final.zip"
extract_path = "/content/dataset"

if os.path.exists(zip_path):
    print("✅ Dataset found! Starting extraction...")
    !unzip -qo "{zip_path}" -d "{extract_path}"
    print("✨ Extraction complete!")
else:
    print("❌ Error: Dataset zip file not found in Drive.")

In [ ]:
# Step 3: Define image transformations and create DataLoaders
# We switched 'RandomResizedCrop' to 'CenterCrop' for the 'train' phase
# to ensure the model focuses consistently on the central wound area.
data_transforms = {
    'train': transforms.Compose([
        # Step A: Resize to 256 to prepare for a clean center crop.
        transforms.Resize(256),

        # Step B: CenterCrop to 224x224 to isolate the gunshot wound.
        # This mirrors the 66% ratio seen in forensic best practices.
        transforms.CenterCrop(224),

        # Step C: Geometric Augmentation.
        # Flips the image horizontally to improve model generalization.
        transforms.RandomHorizontalFlip(),

        # Step D: Convert the image into a PyTorch Tensor.
        transforms.ToTensor(),

        # Step E: Normalize pixel values based on ImageNet standards.
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ]),

    'val': transforms.Compose([
        # Validation remains consistent with CenterCrop to test
        # the model's performance on correctly framed wound images.
        transforms.Resize(256),
        transforms.CenterCrop(224),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ]),
}

data_dir = '/content/dataset'

# Load datasets from the local directory using the modified transforms
image_datasets = {x: datasets.ImageFolder(os.path.join(data_dir, x), data_transforms[x])
                  for x in ['train', 'val']}

# Define DataLoaders (Optimized for T4 GPU with batch_size 32)
dataloaders = {x: torch.utils.data.DataLoader(image_datasets[x],
                                              batch_size=32,
                                              shuffle=True,
                                              num_workers=2)
              for x in ['train', 'val']}

dataset_sizes = {x: len(image_datasets[x]) for x in ['train', 'val']}
class_names = image_datasets['train'].classes

print(f"✅ DataLoaders initialized with Center-Focus strategy.")
print(f"Detected Classes: {class_names}")
print(f"Total Images - Train: {dataset_sizes['train']}, Val: {dataset_sizes['val']}")

In [ ]:
# Step 4: Define the main training and validation loop function
from collections import Counter

# 1. Analyze the dataset distribution using Counter
# It automatically counts how many 'Entrance' and 'Exit' images are in the folder.
train_labels = image_datasets['train'].targets
class_counts = Counter(train_labels)
total_samples = sum(class_counts.values())

# 2. Calculate weights based on the counts
# This ensures the model pays extra attention to the minority class (Exit).
class_weights = [total_samples / (len(class_names) * class_counts[i]) for i in range(len(class_names))]
weights_tensor = torch.tensor(class_weights).to(device).float()

print(f"📊 Dataset Distribution: {dict(class_counts)}")
print(f"⚖️ Automatically calculated weights: {class_weights}")

# 3. Define the main training loop function
def train_model(model, criterion, optimizer, scheduler, num_epochs=25):
    since = time.time()
    best_model_wts = copy.deepcopy(model.state_dict())
    best_acc = 0.0

    for epoch in range(num_epochs):
        print(f'Epoch {epoch}/{num_epochs - 1}')
        print('-' * 15)

        for phase in ['train', 'val']:
            if phase == 'train':
                model.train()
            else:
                model.eval()

            running_loss = 0.0
            running_corrects = 0

            for inputs, labels in dataloaders[phase]:
                inputs, labels = inputs.to(device), labels.to(device)
                optimizer.zero_grad()

                with torch.set_grad_enabled(phase == 'train'):
                    outputs = model(inputs)
                    _, preds = torch.max(outputs, 1)
                    loss = criterion(outputs, labels)

                    if phase == 'train':
                        loss.backward()
                        optimizer.step()

                running_loss += loss.item() * inputs.size(0)
                running_corrects += torch.sum(preds == labels.data)

            if phase == 'train':
                scheduler.step()

            epoch_loss = running_loss / dataset_sizes[phase]
            epoch_acc = running_corrects.double() / dataset_sizes[phase]

            print(f'{phase} Loss: {epoch_loss:.4f} Acc: {epoch_acc:.4f}')

            if phase == 'val' and epoch_acc > best_acc:
                best_acc = epoch_acc
                best_model_wts = copy.deepcopy(model.state_dict())
        print()

    time_elapsed = time.time() - since
    print(f'✅ Training complete in {time_elapsed // 60:.0f}m {time_elapsed % 60:.0f}s')
    print(f'🏆 Best val Acc: {best_acc:4f}')

    model.load_state_dict(best_model_wts)
    return model, best_acc

In [ ]:
# Step 5: Run the comparative benchmark with Weighted Loss (Auto-Weights)
gsw_model_list = ['resnet50', 'efficientnet_b0', 'convnext_tiny']
gsw_results = []

# Directory to save the best model weights on Google Drive
save_path_root = '/content/drive/MyDrive/GSW_Project/Models'
if not os.path.exists(save_path_root):
    os.makedirs(save_path_root)

print("🔥 Starting the Master GSW CNN Benchmark (Weighted Strategy)...")

for model_name in gsw_model_list:
    print(f"\n" + "="*50)
    print(f"Targeting Architecture: {model_name}")
    print("="*50)

    # 1. Initialize model using timm
    model = timm.create_model(model_name, pretrained=True, num_classes=len(class_names)).to(device)

    # 2. [CRITICAL] Apply the calculated 'weights_tensor' from Step 4
    # This ensures each model is punished more for misclassifying 'Exit' wounds.
    criterion = nn.CrossEntropyLoss(weight=weights_tensor)

    # 3. Use AdamW instead of Adam for better stability in modern models
    optimizer = optim.AdamW(model.parameters(), lr=1e-4)
    lr_scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=7, gamma=0.1)

    start_time = time.time()

    # 4. Run the training function defined in Step 4
    # The 'num_epochs' is set to 25 to allow full convergence.
    model, best_acc = train_model(model, criterion, optimizer, lr_scheduler, num_epochs=25)

    duration = time.time() - start_time

    gsw_results.append({
        'Model': model_name,
        'Best_Val_Accuracy': best_acc.item(),
        'Duration_Sec': round(duration, 2)
    })

    # 5. Save best weights for each architecture
    torch.save(model.state_dict(), os.path.join(save_path_root, f'GSW_{model_name}_MASTER.pth'))
    print(f"✅ Saved MASTER weights for {model_name}")

# 6. Display final summary table (The prize for today's work)
print("\n" + "#"*45)
print("  FINAL MASTER GSW BENCHMARK REPORT  ")
print("#"*45)
master_df = pd.DataFrame(gsw_results)
print(master_df)

In [ ]:
# Install the external library for Grad-CAM visualization
!pip install grad-cam

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import torch
import os
import timm
from pytorch_grad_cam import GradCAM
from pytorch_grad_cam.utils.model_targets import ClassifierOutputTarget
from pytorch_grad_cam.utils.image import show_cam_on_image

# 1. Setup Model and Load Master Weights
# Using the class_names and device defined in your previous steps
model_name = 'resnet50'
model = timm.create_model(model_name, pretrained=False, num_classes=len(class_names)).to(device)
save_path_root = '/content/drive/MyDrive/GSW_Project/Models'
checkpoint_path = os.path.join(save_path_root, f'GSW_{model_name}_MASTER.pth')

if os.path.exists(checkpoint_path):
    model.load_state_dict(torch.load(checkpoint_path))
    model.eval()
    print(f"✅ Successfully loaded Master weights for {model_name}")
else:
    print(f"❌ Checkpoint NOT found at: {checkpoint_path}")

# 2. Define Target Layer (The last conv layer of ResNet50)
target_layers = [model.layer4[-1]]
cam = GradCAM(model=model, target_layers=target_layers)

# 3. Select an Image for Visualization
# We use image_datasets['val'] which you just re-initialized in Step 3.
# Let's pick index 10 (you can change this to explore other cases).
index = 3
image, label = image_datasets['val'][index]
input_tensor = image.unsqueeze(0).to(device)

# 4. Generate Heatmap for the True Class
targets = [ClassifierOutputTarget(label)]
grayscale_cam = cam(input_tensor=input_tensor, targets=targets)[0, :]

# 5. De-normalization and Pre-processing for Display
# Convert from (C, H, W) to (H, W, C) and move to CPU
img_np = image.permute(1, 2, 0).cpu().numpy()

# Reverse the ImageNet normalization to see the original colors properly
mean = np.array([0.485, 0.456, 0.406])
std = np.array([0.229, 0.224, 0.225])
img_np = std * img_np + mean
img_np = np.clip(img_np, 0, 1)

# 6. Superimpose the Grad-CAM Heatmap
visualization = show_cam_on_image(img_np, grayscale_cam, use_rgb=True)

# 7. Final Plotting
plt.figure(figsize=(12, 6))

plt.subplot(1, 2, 1)
plt.imshow(img_np)
plt.title(f"Original Image (Actual: {class_names[label]})", fontsize=12)
plt.axis('off')

plt.subplot(1, 2, 2)
plt.imshow(visualization)
plt.title(f"ResNet50 Grad-CAM Visualization", fontsize=12)
plt.axis('off')

plt.tight_layout()
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import torch
import os
import timm
import random
from pytorch_grad_cam import GradCAM
from pytorch_grad_cam.utils.model_targets import ClassifierOutputTarget
from pytorch_grad_cam.utils.image import show_cam_on_image

# 1. Setup EfficientNet and Load Weights
model_name = 'efficientnet_b0'
model = timm.create_model(model_name, pretrained=False, num_classes=len(class_names)).to(device)
save_path_root = '/content/drive/MyDrive/GSW_Project/Models'
checkpoint_path = os.path.join(save_path_root, f'GSW_{model_name}_MASTER.pth')

if os.path.exists(checkpoint_path):
    model.load_state_dict(torch.load(checkpoint_path))
    model.eval()
    print(f"✅ Successfully loaded Master weights for {model_name}")

# 2. Target Layer for EfficientNet
# EfficientNet's spatial features are best captured at 'conv_head'
target_layers = [model.conv_head]
cam = GradCAM(model=model, target_layers=target_layers)

# 3. Visualization Logic (Random selection from validation set)
index = random.randint(0, len(image_datasets['val']) - 1)
image, label = image_datasets['val'][index]
input_tensor = image.unsqueeze(0).to(device)

# 4. Generate Heatmap
targets = [ClassifierOutputTarget(label)]
grayscale_cam = cam(input_tensor=input_tensor, targets=targets)[0, :]

# 5. Reverse Normalization for Display
img_np = image.permute(1, 2, 0).cpu().numpy()
mean = np.array([0.485, 0.456, 0.406])
std = np.array([0.229, 0.224, 0.225])
img_np = std * img_np + mean
img_np = np.clip(img_np, 0, 1)

# 6. Overlay Heatmap
visualization = show_cam_on_image(img_np, grayscale_cam, use_rgb=True)

# 7. Display Results
plt.figure(figsize=(12, 6))

plt.subplot(1, 2, 1)
plt.imshow(img_np)
plt.title(f"Original ({class_names[label]})", fontsize=12)
plt.axis('off')

plt.subplot(1, 2, 2)
plt.imshow(visualization)
plt.title(f"EfficientNet Focus (Grad-CAM)", fontsize=12)
plt.axis('off')

plt.tight_layout()
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import torch
import os
import timm
import random
from pytorch_grad_cam import GradCAM
from pytorch_grad_cam.utils.model_targets import ClassifierOutputTarget
from pytorch_grad_cam.utils.image import show_cam_on_image

# 1. Setup ConvNeXt and Load Master Weights
model_name = 'convnext_tiny'
model = timm.create_model(model_name, pretrained=False, num_classes=len(class_names)).to(device)
save_path_root = '/content/drive/MyDrive/GSW_Project/Models'
checkpoint_path = os.path.join(save_path_root, f'GSW_{model_name}_MASTER.pth')

if os.path.exists(checkpoint_path):
    model.load_state_dict(torch.load(checkpoint_path))
    model.eval()
    print(f"✅ Successfully loaded ConvNeXt Master weights (91.88% Acc)")
else:
    print(f"❌ Checkpoint file not found: {checkpoint_path}")

# 2. Define Target Layer for ConvNeXt
# ConvNeXt is stage-based. We target the first block of the last stage
# to maintain a good balance between spatial resolution and high-level features.
target_layers = [model.stages[-1].blocks[0]]
cam = GradCAM(model=model, target_layers=target_layers)

# 3. Select a Random Image for Visualization
index = random.randint(0, len(image_datasets['val']) - 1)
image, label = image_datasets['val'][index]
input_tensor = image.unsqueeze(0).to(device)

# 4. Generate Grad-CAM Heatmap
targets = [ClassifierOutputTarget(label)]
grayscale_cam = cam(input_tensor=input_tensor, targets=targets)[0, :]

# 5. Reverse Normalization for Accurate Display
img_np = image.permute(1, 2, 0).cpu().numpy()
# ImageNet Stats for De-normalization
mean = np.array([0.485, 0.456, 0.406])
std = np.array([0.229, 0.224, 0.225])
img_np = std * img_np + mean
img_np = np.clip(img_np, 0, 1)

# 6. Overlay Heatmap on Original Image
visualization = show_cam_on_image(img_np, grayscale_cam, use_rgb=True)

# 7. Final Results Display
plt.figure(figsize=(12, 6))

plt.subplot(1, 2, 1)
plt.imshow(img_np)
plt.title(f"Original Image (Actual: {class_names[label]})", fontsize=12)
plt.axis('off')

plt.subplot(1, 2, 2)
plt.imshow(visualization)
plt.title(f"ConvNeXt AI Focus (Grad-CAM)", fontsize=12)
plt.axis('off')

plt.tight_layout()
plt.show()

In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix, classification_report
import os
import timm

# 1. Setup and Path
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
save_path_root = '/content/drive/MyDrive/GSW_Project/Models'
# Ensure class_names are ['Entrance', 'Exit'] or similar
class_names = ['Entrance', 'Exit']

def generate_confusion_matrix(model_name):
    print(f"🔄 Processing Confusion Matrix for: {model_name}")

    # 2. Load Architecture & Master Weights
    model = timm.create_model(model_name, pretrained=False, num_classes=2).to(device)
    checkpoint_path = os.path.join(save_path_root, f'GSW_{model_name}_MASTER.pth')

    if not os.path.exists(checkpoint_path):
        print(f"❌ Error: {checkpoint_path} not found.")
        return

    model.load_state_dict(torch.load(checkpoint_path, map_location=device))
    model.eval()

    all_preds = []
    all_labels = []

    # 3. Get Predictions from Validation Set
    with torch.no_grad():
        for inputs, labels in dataloaders['val']: # Using your dataloader from Step 3
            inputs = inputs.to(device)
            labels = labels.to(device)

            outputs = model(inputs)
            _, preds = torch.max(outputs, 1)

            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    # 4. Calculate Confusion Matrix
    cm = confusion_matrix(all_labels, all_preds)

    # Calculate percentages for better insight
    cm_perc = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis] * 100

    # 5. Plotting
    plt.figure(figsize=(8, 6))
    # Displaying counts and percentages together
    labels_arr = (np.array(["{0:d}\n({1:.1f}%)".format(count, perc)
                 for count, perc in zip(cm.flatten(), cm_perc.flatten())])).reshape(2,2)

    sns.heatmap(cm, annot=labels_arr, fmt="", cmap='Blues',
                xticklabels=class_names, yticklabels=class_names,
                annot_kws={"size": 14, "weight": "bold"})

    plt.xlabel('Predicted Label', fontsize=12)
    plt.ylabel('True Label', fontsize=12)
    plt.title(f'Confusion Matrix: {model_name.upper()}', fontsize=15, pad=20)

    # Save the figure for GitHub/Thesis
    save_filename = f'CM_{model_name}.png'
    plt.savefig(save_filename, dpi=300, bbox_inches='tight')
    plt.show()

    print(f"✅ Saved: {save_filename}")
    print("-" * 30)

    # Optional: Print text-based report for precision/recall
    print(classification_report(all_labels, all_preds, target_names=class_names))

# --- Run for all 3 Models ---
model_list = ['resnet50', 'efficientnet_b0', 'convnext_tiny']

for m in model_list:
    generate_confusion_matrix(m)

---
## 🔍 Interactive Multi-Model Inference Tool (Grad-CAM)

This tool allows for real-time forensic analysis of gunshot wounds using the trained models.
By uploading an image, you can visualize the AI's decision-making process through Grad-CAM heatmaps.

### **How to Use:**
1. **Select Model:** Choose between **ResNet50**, **EfficientNet-B0**, or **ConvNeXt-Tiny** from the dropdown menu.
2. **Upload Image:** Click the 'Upload' button to select a gunshot wound image (JPG/PNG).
3. **Analyze Results:** The AI will display the predicted class (Entrance/Exit), the confidence score, and the Grad-CAM visualization.

> **Note:** This interactive UI is powered by `ipywidgets` and is only functional within an active Jupyter/Colab environment.
---

In [ ]:
"""

import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import transforms
from PIL import Image
import numpy as np
import cv2
import matplotlib.pyplot as plt
import io
import os
import timm
from ipywidgets import FileUpload, Output, VBox, Dropdown, HTML
from IPython.display import display, clear_output
from pytorch_grad_cam import GradCAM
from pytorch_grad_cam.utils.model_targets import ClassifierOutputTarget
from pytorch_grad_cam.utils.image import show_cam_on_image

# --- 1. Configuration & Path Setup ---
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
save_path_root = '/content/drive/MyDrive/GSW_Project/Models'
class_names = ['Entrance', 'Exit']

# --- 2. Target Layer Mapping ---
def get_target_layer(model, name):
    if 'resnet50' in name: return [model.layer4[-1]]
    elif 'efficientnet' in name: return [model.conv_head]
    elif 'convnext' in name: return [model.stages[-1].blocks[0]]

# --- 3. UI Components ---
model_selector = Dropdown(
    options=[('ResNet50 (Reliable)', 'resnet50'),
             ('EfficientNet-B0 (Fast)', 'efficientnet_b0'),
             ('ConvNeXt-Tiny (SOTA 91.8%)', 'convnext_tiny')],
    value='convnext_tiny',
    description='Select AI:',
)

uploader = FileUpload(accept='image/*', multiple=False)
output_area = Output()

header_html = HTML("<h2>🩸 GSW Forensic Multi-Model Analyzer</h2><p>Select a model and upload an image for instant classification and Grad-CAM focus analysis.</p>")

# --- 4. Main Analysis Logic ---
def run_analysis(change):
    with output_area:
        clear_output()
        if not uploader.value: return

        selected_model_name = model_selector.value
        print(f"⚙️ Loading {selected_model_name} Master weights...")

        # Load Selected Model
        try:
            model = timm.create_model(selected_model_name, pretrained=False, num_classes=2).to(device)
            checkpoint_path = os.path.join(save_path_root, f'GSW_{selected_model_name}_MASTER.pth')
            model.load_state_dict(torch.load(checkpoint_path, map_location=device))
            model.eval()
        except Exception as e:
            print(f"❌ Error loading model: {e}")
            return

        # Process Uploaded Image
        file_info = list(uploader.value.values())[0] if isinstance(uploader.value, dict) else uploader.value[0]
        image_raw = Image.open(io.BytesIO(file_info['content'])).convert('RGB')

        # Preprocessing (Matching the CenterCrop strategy used in training)
        preprocess = transforms.Compose([
            transforms.Resize(256),
            transforms.CenterCrop(224),
            transforms.ToTensor(),
            transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
        ])
        input_tensor = preprocess(image_raw).unsqueeze(0).to(device)

        # 1. Prediction & Confidence
        with torch.no_grad():
            output = model(input_tensor)
            probs = F.softmax(output, dim=1)
            conf, pred_idx = torch.max(probs, dim=1)
            confidence = conf.item() * 100
            result_label = class_names[pred_idx.item()]

        # 2. Grad-CAM Generation
        target_layers = get_target_layer(model, selected_model_name)
        cam = GradCAM(model=model, target_layers=target_layers)
        targets = [ClassifierOutputTarget(pred_idx.item())]

        grayscale_cam = cam(input_tensor=input_tensor, targets=targets)[0, :]

        # 3. Visualization Preparation
        # De-normalize for visualization
        img_display = np.array(image_raw.resize((256, 256))) # Simple resize for background
        # Center crop the display image to match input_tensor spatial area
        h, w, _ = img_display.shape
        start_h, start_w = (h - 224)//2, (w - 224)//2
        img_display = img_display[start_h:start_h+224, start_w:start_w+224]
        img_display = img_display / 255.0

        visualization = show_cam_on_image(img_display, grayscale_cam, use_rgb=True)

        # 4. Final Display
        fig, ax = plt.subplots(1, 2, figsize=(16, 8))

        ax[0].imshow(img_display)
        ax[0].set_title("Original (Center-Focused)", fontsize=14)
        ax[0].axis('off')

        ax[1].imshow(visualization)
        title_color = 'blue' if result_label == 'Entrance' else 'red'

        # --- 수정된 부분: 모델 이름을 타이틀에 추가 ---
        title_text = f"[{selected_model_name.upper()}]\nResult: {result_label} ({confidence:.2f}%)"
        ax[1].set_title(title_text, fontsize=18, color=title_color, fontweight='bold')
        # ------------------------------------------

        ax[1].axis('off')

        plt.tight_layout()
        plt.show()

        print(f"📊 Forensic Report: The {selected_model_name} model identifies this as an '{result_label}' wound with {confidence:.2f}% confidence.")

# --- 5. UI Event Binding ---
uploader.observe(run_analysis, names='value')

# --- 6. Launch ---
display(VBox([header_html, model_selector, uploader, output_area]))

"""